In [2]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

c:\Users\ASUS\OneDrive\Desktop\DeepForge\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv()

True

In [4]:
llm=ChatGoogleGenerativeAI(
    model= "gemini-2.0-flash",
    temperature=0,
    max_tokens= None,
    timeout= None,
    max_retries= 2,
)

In [5]:
response= llm.invoke("Say 'DeepForge Initialized' if you can hear me.")
print(response.content)

DeepForge Initialized


## arXiv Retriever

In [7]:
import arxiv

def search_and_download_paper(query, download_path="../data/"):
    """Searches arXiv for a paper and downloads the PDF."""
    # Ensure the data directory exists
    os.makedirs(download_path, exist_ok=True)

    search= arxiv.Search(
        query= query,
        max_results= 1,
        sort_by= arxiv.SortCriterion.Relevance
    )

    # Fetch the result
    paper = next(search.results())

    filename= f"{paper.title.replace('', '_').replace(':', '')}.pdf"
    print(f"Found: {paper.title}")
    print(f"Authors: {', '.join(author.name for author in paper.authors)}")

    # Download
    path = paper.download_pdf(dirpath=download_path, filename=filename)
    print(f"✅ Downloaded to: {path}")
    return path

# Testing with our reference example
paper_path = search_and_download_paper("QLoRA: Efficient Finetuning of LLMs")

C:\Users\ASUS\AppData\Local\Temp\ipykernel_3192\3278410609.py:15: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  paper = next(search.results())


Found: QLoRA: Efficient Finetuning of Quantized LLMs
Authors: Tim Dettmers, Artidoro Pagnoni, Ari Holtzman, Luke Zettlemoyer
✅ Downloaded to: ../data/_Q_L_o_R_A__ _E_f_f_i_c_i_e_n_t_ _F_i_n_e_t_u_n_i_n_g_ _o_f_ _Q_u_a_n_t_i_z_e_d_ _L_L_M_s_.pdf


In [8]:
import fitz

def extract_text_from_pdf(path):
    """Opens a PDF and extracts all text content."""
    doc= fitz.open(path)
    content= ""

    for page_num in range(len(doc)):
        page= doc.load_page(page_num)
        content += f"\n--- Page {page_num + 1} ---\n"
        content += page.get_text()

    print(f"Extracted {len(content)} characters from {len(doc)} pages.")
    return content

# Extract the text from the downloaded QLoRA paper
raw_text = extract_text_from_pdf(paper_path)

# Preview the start of the paper
print("\n--- Start of Paper ---")
print(raw_text[:1000] + "...")

Extracted 88228 characters from 26 pages.

--- Start of Paper ---

--- Page 1 ---
QLORA: Efficient Finetuning of Quantized LLMs
Tim Dettmers∗
Artidoro Pagnoni∗
Ari Holtzman
Luke Zettlemoyer
University of Washington
{dettmers,artidoro,ahai,lsz}@cs.washington.edu
Abstract
We present QLORA, an efficient finetuning approach that reduces memory us-
age enough to finetune a 65B parameter model on a single 48GB GPU while
preserving full 16-bit finetuning task performance. QLORA backpropagates gradi-
ents through a frozen, 4-bit quantized pretrained language model into Low Rank
Adapters (LoRA). Our best model family, which we name Guanaco, outperforms
all previous openly released models on the Vicuna benchmark, reaching 99.3%
of the performance level of ChatGPT while only requiring 24 hours of finetuning
on a single GPU. QLORA introduces a number of innovations to save memory
without sacrificing performance: (a) 4-bit NormalFloat (NF4), a new data type that
is information theoretically optimal